#1) Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

#2) Load Archives

In [ ]:
# Carregar os dados
ratings = pd.read_csv('/content/ratings.csv')
movies = pd.read_csv('/content/movies.csv')

#3) Best Seller

## Indicates the most viewed and best-rated film in a weighted manner.

In [ ]:
def best_seller_recommendations_weighted_time(ratings, movies, top_n=10, alpha=0.7, decay=0.001):
    # Convert timestamp to datetime and calculate rating age in days
    ratings = ratings.copy()
    ratings['date'] = pd.to_datetime(ratings['timestamp'], unit='s')
    most_recent = ratings['date'].max()
    ratings['days_old'] = (most_recent - ratings['date']).dt.days

    # Temporal weighting: recent ratings carry more weight
    ratings['time_weight'] = np.exp(-decay * ratings['days_old'])

    # Calculate time-weighted average stats
    def weighted_stats(group):
        weighted_avg = (group['rating'] * group['time_weight']).sum() / group['time_weight'].sum()
        return pd.Series({
            'avg_rating_time_weighted': weighted_avg,
            'avg_rating': group['rating'].mean(),
            'num_ratings': len(group)
        })

    movie_stats = ratings.groupby('movieId').apply(weighted_stats).reset_index()

    min_ratings = 10
    movie_stats = movie_stats[movie_stats['num_ratings'] >= min_ratings]

    # Bayesian weighted rating using the time-weighted average
    C = movie_stats['avg_rating_time_weighted'].mean()
    m = min_ratings

    movie_stats['weighted_rating'] = (
        (movie_stats['num_ratings'] / (movie_stats['num_ratings'] + m)) * movie_stats['avg_rating_time_weighted'] +
        (m / (movie_stats['num_ratings'] + m)) * C
    )

    # Sort and return top recommendations
    recommendations = movie_stats.sort_values(by='weighted_rating', ascending=False).head(top_n)
    return recommendations.merge(movies, on='movieId')

In [ ]:
top_n=50
print(f"The top {top_n} movies are:\n")
best_seller_recommendations_weighted_time(ratings, movies, top_n)

The top 50 movies are:



/tmp/ipykernel_29221/1093561914.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  movie_stats = ratings.groupby('movieId').apply(weighted_stats).reset_index()


,movieId,avg_rating_time_weighted,avg_rating,num_ratings,weighted_rating,title,genres
0,318,4.405424,4.429022,317.0,4.372412,"Shawshank Redemption, The (1994)",Crime|Drama
1,3703,4.619316,4.037500,40.0,4.360634,"Road Warrior, The (Mad Max 2) (1981)",Action|Adventure|Sci-Fi|Thriller
2,1172,4.624752,4.161765,34.0,4.329559,Cinema Paradiso (Nuovo cinema Paradiso) (1989),Drama
3,194,4.809252,3.666667,18.0,4.279485,Smoke (1995),Comedy|Drama
4,1209,4.801033,4.305556,18.0,4.274202,Once Upon a Time in the West (C'era una volta ...,Action|Drama|Western
5,1258,4.342149,4.082569,109.0,4.256751,"Shining, The (1980)",Horror
6,3508,4.746164,4.250000,18.0,4.238929,"Outlaw Josey Wales, The (1976)",Action|Adventure|Drama|Thriller|Western
7,296,4.258171,4.197068,307.0,4.228762,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
8,3468,4.713384,4.333333,18.0,4.217856,"Hustler, The (1961)",Drama
9,1221,4.286363,4.259690,129.0,4.217266,"Godfather: Part II, The (1974)",Crime|Drama


#4) Movie Similarity

In [ ]:
def movie_similarity_weighted(movie_id, ratings, movies, top_n=10, alpha=0.7):
    # Create the user-item matrix
    user_movie_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating').fillna(0)

    # Calculate Cosine Similarity between movies
    cosine_sim = cosine_similarity(user_movie_matrix.T)
    cosine_sim_df = pd.DataFrame(cosine_sim, index=user_movie_matrix.columns, columns=user_movie_matrix.columns)

    # Get similarity scores for the target movie
    similar_movies = cosine_sim_df[movie_id].sort_values(ascending=False)
    similar_movies = similar_movies.drop(movie_id)
    similar_movies.name = 'similarity'

    # Merge similarity data with movie titles/genres
    similar_movies = similar_movies.reset_index().merge(movies, on='movieId')

    # Calculate Bayesian Weighted Rating
    movie_stats = ratings.groupby('movieId').agg(
        avg_rating=('rating', 'mean'),
        num_ratings=('rating', 'count')
    ).reset_index()

    m = 10  # Minimum ratings threshold
    C = movie_stats['avg_rating'].mean()  # Global average rating

    movie_stats['weighted_rating'] = (
        (movie_stats['num_ratings'] / (movie_stats['num_ratings'] + m)) * movie_stats['avg_rating'] +
        (m / (movie_stats['num_ratings'] + m)) * C
    )

    # Merge weighted rating with the similarity results
    similar_movies = similar_movies.merge(movie_stats[['movieId', 'weighted_rating', 'avg_rating', 'num_ratings']], on='movieId', how='left')

    # Normalize both metrics to a [0, 1] scale for fair comparison
    similar_movies['similarity_norm'] = (
        similar_movies['similarity'] / similar_movies['similarity'].max()
    )
    similar_movies['weighted_rating_norm'] = (
        similar_movies['weighted_rating'] / similar_movies['weighted_rating'].max()
    )

    # Calculate final weighted score
    # Alpha controls the balance between similarity and overall quality/popularity
    similar_movies['final_score'] = (
        alpha * similar_movies['similarity_norm'] +
        (1 - alpha) * similar_movies['weighted_rating_norm']
    )

    # Sort by final score and return top results
    similar_movies = similar_movies.sort_values(by='final_score', ascending=False)

    return similar_movies[['movieId', 'final_score', 'similarity', 'weighted_rating', 'avg_rating', 'num_ratings', 'title', 'genres']].head(top_n)

In [ ]:
movie_id = 1
print(f"{movies[movies['movieId'] == movie_id]['title'].iloc[0]} has the following similar films:\n")
movie_similarity_weighted(movie_id, ratings, movies, top_n=50, alpha=0.7)

Toy Story (1995) has the following similar films:



,movieId,final_score,similarity,weighted_rating,avg_rating,num_ratings,title,genres
3,260,0.967787,0.557388,4.193964,4.231076,251,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
0,3114,0.959818,0.572601,3.804902,3.860825,97,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
4,356,0.951352,0.547096,4.137535,4.164134,329,Forrest Gump (1994),Comedy|Drama|Romance|War
1,480,0.946213,0.565637,3.730341,3.750000,238,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller
6,1210,0.941122,0.541089,4.095264,4.137755,196,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi
5,364,0.928167,0.541145,3.904530,3.941860,172,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX
2,780,0.924494,0.564262,3.436908,3.445545,202,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller
18,318,0.921691,0.508545,4.393347,4.429022,317,"Shawshank Redemption, The (1994)",Crime|Drama
9,1270,0.921196,0.530381,3.995163,4.038012,171,Back to the Future (1985),Adventure|Comedy|Sci-Fi
8,1265,0.919295,0.534169,3.899506,3.944056,143,Groundhog Day (1993),Comedy|Fantasy|Romance


Similaridade de usuário

In [ ]:
def user_similarity_weighted(user_id, ratings, movies, top_n=10, alpha=0.7):
    # Create the user-item matrix
    user_movie_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating').fillna(0)

    # Calculate Cosine Similarity between users
    user_similarity = cosine_similarity(user_movie_matrix)
    user_similarity_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)

    # Get the most similar users to the target user
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)
    similar_users = similar_users.drop(user_id)

    # Get movies the target user has already rated (to exclude them)
    user_rated_movies = set(ratings[ratings['userId'] == user_id]['movieId'])

    # Filter ratings from similar users for movies not seen by the target user
    recommendations = ratings[
        ratings['userId'].isin(similar_users.index) &
        ~ratings['movieId'].isin(user_rated_movies)
    ].copy()

    # Map the similarity weight of each user to the recommendations
    recommendations['user_similarity'] = recommendations['userId'].map(similar_users)

    # Calculate average rating weighted by user similarity
    # (More similar users have a higher impact on the score)
    def weighted_avg(group):
        avg = (group['rating'] * group['user_similarity']).sum() / group['user_similarity'].sum()
        return pd.Series({
            'weighted_avg_rating': avg,
            'num_ratings': len(group)
        })

    recommendations = recommendations.groupby('movieId').apply(weighted_avg).reset_index()

    # Filter movies with a minimum number of ratings
    min_ratings = 5
    recommendations = recommendations[recommendations['num_ratings'] >= min_ratings]

    # Calculate global Bayesian Weighted Rating
    movie_stats = ratings.groupby('movieId').agg(
        avg_rating=('rating', 'mean'),
        total_ratings=('rating', 'count')
    ).reset_index()

    m = 10  # Minimum ratings threshold
    C = movie_stats['avg_rating'].mean()  # Global average

    movie_stats['weighted_rating'] = (
        (movie_stats['total_ratings'] / (movie_stats['total_ratings'] + m)) * movie_stats['avg_rating'] +
        (m / (movie_stats['total_ratings'] + m)) * C
    )

    # Merge global weighted rating with the user-similarity recommendations
    recommendations = recommendations.merge(movie_stats[['movieId', 'weighted_rating', 'avg_rating', 'total_ratings']], on='movieId', how='left')

    # Normalize metrics to a [0, 1] scale
    recommendations['weighted_avg_norm'] = (
        recommendations['weighted_avg_rating'] / recommendations['weighted_avg_rating'].max()
    )
    recommendations['weighted_rating_norm'] = (
        recommendations['weighted_rating'] / recommendations['weighted_rating'].max()
    )

    # Calculate final hybrid score
    recommendations['final_score'] = (
        alpha * recommendations['weighted_avg_norm'] +
        (1 - alpha) * recommendations['weighted_rating_norm']
    )

    # Sort results and merge with movie metadata
    recommendations = recommendations.sort_values(by='final_score', ascending=False)
    recommendations = recommendations.merge(movies, on='movieId')

    return recommendations[['movieId', 'final_score', 'weighted_avg_rating', 'weighted_rating', 'avg_rating', 'num_ratings', 'title', 'genres']].head(top_n)

In [ ]:
movie_id = 1
print(f"{movies[movies['movieId'] == movie_id]['title'].iloc[0]} has the following similar user recommendations:\n")
user_similarity_weighted(movie_id, ratings, movies, top_n=50, alpha=0.7)

Toy Story (1995) has the following similar user recommendations:



/tmp/ipykernel_29221/2023918256.py:28: RuntimeWarning: invalid value encountered in scalar divide
  avg = (group['rating'] * group['user_similarity']).sum() / group['user_similarity'].sum()
/tmp/ipykernel_29221/2023918256.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  recommendations = recommendations.groupby('movieId').apply(weighted_avg).reset_index()


,movieId,final_score,weighted_avg_rating,weighted_rating,avg_rating,num_ratings,title,genres
0,6460,0.960050,4.863884,3.808299,4.900000,5.0,"Trial, The (Procès, Le) (1962)",Drama
1,177593,0.942586,4.687839,3.923582,4.750000,8.0,"Three Billboards Outside Ebbing, Missouri (2017)",Crime|Drama
2,318,0.939777,4.445433,4.393347,4.429022,317.0,"Shawshank Redemption, The (1994)",Crime|Drama
3,1178,0.933671,4.608519,3.960204,4.541667,12.0,Paths of Glory (1957),Drama|War
4,2239,0.933442,4.688144,3.789030,4.666667,6.0,Swept Away (Travolti da un insolito destino ne...,Comedy|Drama
5,31364,0.921725,4.629221,3.741632,4.700000,5.0,Memories of Murder (Salinui chueok) (2003),Crime|Drama|Mystery|Thriller
6,1104,0.921155,4.469071,4.070816,4.475000,20.0,"Streetcar Named Desire, A (1951)",Drama
7,858,0.920063,4.382042,4.238240,4.289062,192.0,"Godfather, The (1972)",Crime|Drama
8,92535,0.917838,4.583426,3.781224,4.300000,10.0,Louis C.K.: Live at the Beacon Theater (2011),Comedy
9,1041,0.916816,4.492299,3.958309,4.590909,11.0,Secrets & Lies (1996),Drama
